In [ ]:
# Step 3 — Color–Magnitude Diagram (RUWE-colored)
#
# Purpose
#   - Load a final member catalog (CSV) produced by Step 1.
#   - Clean CMD columns and RUWE.
#   - Plot a Gaia CMD (G vs. BP-RP) colored by RUWE, with high-RUWE points drawn last.
#
# Notes
#   - "phot_g_mean_mag" is an apparent G magnitude unless you already converted it.
#   - This step colors by RUWE only; selection/cuts happen in later steps.

from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
@dataclass
class RUWECMDConfig:
    cluster_name: str
    input_members_csv: str

    # Column names (expected in your members CSV)
    color_col: str = "bp_rp"
    mag_col: str = "phot_g_mean_mag"
    ruwe_col: str = "ruwe"

    # Optional: compute absolute magnitude using a fixed distance modulus (m - M)
    # Set to None to plot magnitude as-is.
    distance_modulus: Optional[float] = None

    # Optional: apply a magnitude range cut (useful to control outliers)
    mag_limits: Optional[Tuple[float, float]] = None  # e.g., (8, 20)

    # RUWE display range (controls contrast in the colormap)
    ruwe_vmin: float = 0.8
    ruwe_vmax: float = 1.4

    # Plot styling
    figsize: Tuple[float, float] = (10.0, 8.0)
    marker_size: float = 3.0
    alpha: float = 0.8
    colormap: str = "turbo"

    # Output (optional)
    save_path: Optional[str] = None  # e.g., "M67_CMD_RUWE.png"
    dpi: int = 300


# ---------------------------------------------------------------------
# Core functionality
# ---------------------------------------------------------------------
def load_members(path: str) -> pd.DataFrame:
    path_obj = Path(path)
    if not path_obj.exists():
        raise FileNotFoundError(f"Members file not found: {path_obj.resolve()}")
    return pd.read_csv(path_obj)


def prepare_ruwe_cmd_data(df: pd.DataFrame, cfg: RUWECMDConfig) -> pd.DataFrame:
    df = df.copy()

    # Coerce to numeric
    for col in (cfg.color_col, cfg.mag_col, cfg.ruwe_col):
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Drop NaNs
    cmd = df.dropna(subset=[cfg.color_col, cfg.mag_col, cfg.ruwe_col]).copy()

    # Optional magnitude cut
    if cfg.mag_limits is not None:
        mag_min, mag_max = cfg.mag_limits
        cmd = cmd[(cmd[cfg.mag_col] >= mag_min) & (cmd[cfg.mag_col] <= mag_max)].copy()

    # Optional absolute magnitude conversion using fixed distance modulus
    if cfg.distance_modulus is not None:
        cmd["plot_mag"] = cmd[cfg.mag_col] - float(cfg.distance_modulus)
    else:
        cmd["plot_mag"] = cmd[cfg.mag_col]

    # Sort so high RUWE is drawn last (improves visibility)
    cmd = cmd.sort_values(cfg.ruwe_col, ascending=True)

    return cmd


def plot_ruwe_cmd(cmd: pd.DataFrame, cfg: RUWECMDConfig) -> None:
    plt.figure(figsize=cfg.figsize)

    sc = plt.scatter(
        cmd[cfg.color_col].values,
        cmd["plot_mag"].values,
        s=cfg.marker_size,
        c=cmd[cfg.ruwe_col].values,
        cmap=cfg.colormap,
        vmin=cfg.ruwe_vmin,
        vmax=cfg.ruwe_vmax,
        alpha=cfg.alpha,
    )

    plt.gca().invert_yaxis()
    plt.xlabel(r"Color ($G_{\rm BP} - G_{\rm RP}$)")
    ylabel = r"Absolute Magnitude ($G$)" if cfg.distance_modulus is not None else r"Apparent Magnitude ($G$)"
    plt.ylabel(ylabel)
    plt.title(f"{cfg.cluster_name} CMD Colored by RUWE")
    plt.grid(True, alpha=0.3)

    cbar = plt.colorbar(sc)
    cbar.set_label("Gaia RUWE", fontsize=11)

    plt.tight_layout()

    if cfg.save_path is not None:
        out_path = Path(cfg.save_path)
        plt.savefig(out_path, dpi=cfg.dpi)
        print(f"Saved RUWE CMD figure: {out_path.resolve()}")

    plt.show()


def main(cfg: RUWECMDConfig) -> None:
    df = load_members(cfg.input_members_csv)
    print(f"Loaded members: {len(df)} rows")

    cmd = prepare_ruwe_cmd_data(df, cfg)
    print(f"CMD-ready sample: {len(cmd)} rows")

    plot_ruwe_cmd(cmd, cfg)


# ---------------------------------------------------------------------
# User configuration (edit here)
# ---------------------------------------------------------------------
cfg = RUWECMDConfig(
    cluster_name="M67",
    input_members_csv="M67_Members_Final.csv",
    distance_modulus=None,           # set numeric value if you want absolute magnitude via DM
    mag_limits=None,                 # e.g., (8, 20)
    ruwe_vmin=0.8,
    ruwe_vmax=1.4,
    save_path=None,                  # e.g., "M67_CMD_RUWE.png"
)

main(cfg)
